In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Get Data
tickers = ["AAPL", "MSFT", "GOOG", "NVDA", 
            "JPM", "BAC", "GS", "WFC",
           "XOM", "CVX", "COP",
           "WMT", "JNJ", "TGT", "AMZN"]
data = yf.download(tickers, start = "2020-01-01", end = "2026-01-01")["Close"]
daily_returns = data.pct_change().dropna()

In [ ]:
#Correlation Heatmap
corr = daily_returns.corr()
sns.heatmap(corr, center = 0, cmap = "Blues")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
#Scaling to prevent PC's from reflecting the highest variance
scaler = StandardScaler()
scaled_returns = scaler.fit_transform(daily_returns)
pca = PCA()
pca.fit(scaled_returns)
#Eigenvalues = variance explained by each PC
eigenvalues = pca.explained_variance_
#Explained variance ratio
explained_var_ratio = pca.explained_variance_ratio_
#Cumulative Variance
cumulative_var = np.cumsum(explained_var_ratio)
components = [f"PC{i+1}" for i in range (len(eigenvalues))]

results_df = pd.DataFrame({"Components" : components, 
                          "Eigenvalues":eigenvalues,
                          "Explained Variance %": explained_var_ratio * 100,
                          "Cumulative Variance %": cumulative_var * 100})
print(results_df.round(2))                                                  

In [ ]:
#Visualization
#Top Graph(Components & Explained Variance pdf)
plt.bar(components, explained_var_ratio)
plt.xlabel("PC")
plt.xticks(fontsize = 8)
plt.ylabel("Variance Explained %")
plt.title("Variance Explained by each PC")
plt.show()

#Bottom Graph(Components & Cumulative Variance cdf)
plt.plot(components, cumulative_var)
plt.axhline(0.8, color = "red", linestyle = "--")
plt.xlabel("PC")
plt.xticks(fontsize = 8)
plt.ylabel("Cumulative Variance %")
plt.title("Cumulative Variance Explained")
plt.show()

In [ ]:
#Kaiser Criterion: Only keeps PC's w/ eigenvalues > 1
#Captures the most important PC's
def filter_eigenvalues(components):
    selected_eigenvalues = []
    for eigenvalue in components:
        if eigenvalue >= 1:
            selected_eigenvalues.append(eigenvalue)
    return selected_eigenvalues

kaiser_components = filter_eigenvalues(eigenvalues)
print(f"Eigenvalues > 1: {kaiser_components}")

In [ ]:
#Factor Loadings
eigenvectors = pca.components_
PC_names = [f"PC{i+1}" for i in range(len(eigenvalues))]
transposed_eigenvectors = pd.DataFrame(eigenvectors.T, 
                                       columns = PC_names, 
                                       index = tickers)
#Adjust Loadings
loadings = transposed_eigenvectors * np.sqrt(eigenvalues)

In [ ]:
#Visualize Loadings
#PC1 = Beta(Market Factor)
#PC2 = Growth vs Value Stocks
#PC3 = Consumer Sector
#PC4 = Unclear (Maybe Energy Sector)
loadings_subset = loadings.iloc[:,:4]
sns.heatmap(loadings_subset, center = 0, annot = True, cmap = "RdBu_r")
plt.title("First 4 PC's Correlation")
plt.tight_layout()
plt.show()

In [ ]:
#Monte Carlo Simulation
results = []
for i in range(1000):
    bootstrap_sample = daily_returns.sample(frac = 1, replace = True)
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(bootstrap_sample)
    
    pca = PCA()
    pca.fit(scaled_data)
    eigenvalues = pca.explained_variance_

    results.append(eigenvalues)

results_df = pd.DataFrame(results)

In [ ]:
#Summary Statistics
#Mean
bootstrapped_mean = results_df.mean()
#5th Percentile
fifth_percentile = results_df.quantile(0.05)
#95th Percentile
ninetyfifth_percentile = results_df.quantile(0.95)
summary = pd.DataFrame({"Mean": bootstrapped_mean,
                        "5th Percentile": fifth_percentile,
                        "95th Percentile": ninetyfifth_percentile})
print(summary.iloc[:, :4])
                        

#All Eigenvalue Stability
PCs = results_df.iloc[:, :4]
plt.boxplot(PCs)
plt.title("Eigenvalue Stability Across 1000 Bootstrapped Samples")
plt.xlabel("Components")
plt.xticks([1,2,3,4], ["PC1", "PC2", "PC3", "PC4"])
plt.ylabel("Eigenvalues")
plt.axhline(1, linestyle = "--", color = "red")
plt.show()

#PC1 Stability
PC1 = results_df.iloc[:, :1]
plt.hist(PC1)
plt.title("PC1 Stability Across 1000 Bootstrapped Samples")
plt.xlabel("PC1 Eigenvalue")
plt.axvline(1, linestyle = "--", color = "red")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

#PC4 Stability
PC4 = results_df.iloc[:, 3:4]
plt.hist(PC4)
plt.title("PC4 Stability Across 1000 Bootstrapped Samples")
plt.xlabel("PC4 Eigenvalue")
plt.axvline(1, linestyle = "--", color = "red")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()